<font color=#009afa size=50>DAT-6002 - Fundamentals of Artificial Intelligence</font><br>
<font size=50>WRIT1 Part 2 - Neural Networks</font><br>
<font size=20 color=#00c382>MLP using SciKitLearn - Bank Data</font><br>

|**Assessment Code:** |WRIT1|
|---|---|
|**Author:**|Tom Rowan|
|**Course:**|DAT6002_S2_25|
|**Course Tutor:**|Imtiaz Hussain Khan, PhD|


**Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.**

https://archive.ics.uci.edu/dataset/222/bank+marketing

---

# Initialise Notebook

In [ ]:
# Install Requirements Using Pip
%pip install -r requirements.txt

### Import Python Libraries

In [ ]:
# Standard Libraries
import sys
import time
import json
import random
from datetime import datetime

# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
from matplotlib import colormaps
import matplotlib.colors as mcolors
from matplotlib.pyplot import figure
import matplotlib.ticker as mticker
import matplotlib.animation as animation
%matplotlib inline

# Seaborn Graphs
import seaborn as sns

# Plotly
import plotly.express as px
import plotly.graph_objects as go

# Tabulate
from tabulate import tabulate

# Colorama
from colorama import Fore, Back, Style

# Requests
import requests

# Pycountry 
import pycountry

# GeoPy
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Cartopy Maps
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Geopandas
import geopandas as gpd

# IPython Display 
from IPython.display import HTML

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Create a Timestamp

In [ ]:
# Generate Timestamp
today = pd.Timestamp(datetime.today())
now_date_timestamp = str(today.strftime("%Y%m%d"))
now_datetime_timestamp = str(today.strftime("%Y%m%d-%H%M"))

### Functions for Pretty Output

print_output - Print coloured Output

In [ ]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow', newline=True):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
        
    # Newline - defaul to a newline
    nl=None if newline else ' '
    
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}', end=nl)

### Check for Internet Access

In [ ]:
_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.github.com',
                  'https://datascience.daemonchild.com']

Set this to False if you have a local dataset

In [ ]:
internet_required = False

In [ ]:
# Function to Safely Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online

# Perform the checks
if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    online = False


# Stop running if Internet is required
if online == False & internet_required == True:
    sys.exit('Stopping notebook run here.')    


---
# Load the Dataset

In [ ]:
if online:
    # Fetch the dataset from the Internet
    df = pd.read_csv('bank_full_scaled_latest.csv', delimiter=',')
else:
    # Load the dataset from local file
    df = pd.read_csv('bank_full_scaled_latest.csv', delimiter=',')

In [ ]:
df.sample(10)

Now it is necessary to split the input dataset into two parts - testing and training data

In [ ]:
# Split data frame into 80/20
df_train = df.sample(frac = 0.8)
# Creating dataframe with rest of the 20% values
df_test = df.drop(df_train.index)

In [ ]:
print_output("df_train shape", df_train.shape)
print_output("df_test shape", df_test.shape)

In [ ]:
# Split datasets into X (features) and y (labels)
df_X_train = df_train.drop(columns='y')
df_y_train = df_train['y']

df_X_test = df_test.drop(columns='y')
df_y_test = df_test['y']


In [ ]:
df_X_train.columns

# Neural Network

"Construct a three-layer feedforward neural network consisting of N input neurons, 10 hidden neurons, and M output neuron(s)."

In a neural network, the forward pass consists of two main operations per layer:

    Linear Transformation: Z=X⋅W+b

    Activation Function: A=σ(Z)

For a binary classification task like this, the Sigmoid function is the standard choice for the output layer because it squashes any input into a value between 0 and 1, which we can interpret as a probability.  (citation)

## MLP Class 

In [ ]:
class MLP_From_Scratch:

    # Constructor
    def __init__(self, X_train, y_train, X_test, y_test, **kwargs):

        # Handle Arguments
        _random_seed = kwargs.get('random_seed', 67)
        self.debug = kwargs.get('debug', False)

        self.validation_split = kwargs.get('validation_split', 0.2)    # Default to 20% of the training data

        # Split off validation dataset from the training data
        val_X = X_train.sample(frac=self.validation_split, random_state=_random_seed)
        val_y = y_train.loc[val_X.index]

        # Convert provided data to numpy arrays
        self.X_np_train = X_train.to_numpy()
        self.y_np_train = y_train.to_numpy().reshape(-1,1)

        self.X_np_val = val_X.to_numpy()
        self.y_np_val = val_y.to_numpy().reshape(-1,1)

        self.X_np_test = X_test.to_numpy()
        self.y_np_test = y_test.to_numpy().reshape(-1,1)

        # Additional Model Parameters
        _act_func = kwargs.get('act_func', 'sigmoid')
        _no_hidden_neurons = kwargs.get('no_hidden_neurons', 10)    # Defaults to 10 hidden neurons, per assessment specification
        _no_output_neurons = kwargs.get('no_output_neurons', 1)

        # Define Model Parameters
        self.no_input_neurons  = self.X_np_train.shape[1]    # Set dynamically based on the input dataset
        self.no_hidden_neurons = _no_hidden_neurons          
        self.no_output_neurons = _no_output_neurons
        self.learning_rate = 0.03          

        # Initialize Weights with small random numbers
        # 
        #   input_neurons --> weights_0 --> hidden_neurons --> weights_1 --> output_neurons
        #                                   biases_1                         biases_2
        #
        np.random.seed(_random_seed)

        self.weights_0 = np.random.randn(self.no_input_neurons, self.no_hidden_neurons) 
        self.weights_1 = np.random.randn(self.no_hidden_neurons, self.no_output_neurons)

        self.biases_1 = np.zeros((1, self.no_hidden_neurons))
        self.biases_2 = np.zeros((1, self.no_output_neurons))


        if self.debug:
            print_output("No Input Layer Neurons", self.no_input_neurons, colour='blue')
            print_output("No Input to Hidden Weights", self.weights_0.size)

            print_output("No Hidden Layer Neurons", self.no_hidden_neurons, colour='blue')
            print_output("No Hidden Layer Biases", self.biases_1.size, colour='blue')

            print_output("No Hidden to Output Weights", self.weights_1.size)

            print_output("No Output Layer Neurons", self.no_output_neurons, colour='blue')
            print_output("No Output Layer Biases", self.biases_2.size, colour='blue')


    # end Constructor

    # ***** Activation Functions 

    def _relu(self, x):
        return np.maximum(0, x)

    def _relu_derivative(self, x):
        return (x > 0).astype(float)

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def _sigmoid_derivative(self, x):
        return x * (1 - x)

    # Generic version to allow replacement
    def _act_func(self, x):
        return self._sigmoid(x)

    def _act_func_deriv(self, x):
        return self._sigmoid_derivative(x)
    
    # ***** MLP Process Functions 

    def _feed_forward(self, X):
        sum_synapse_0 = np.dot(X, self.weights_0) + self.biases_1
        hidden_layer = self._act_func(sum_synapse_0)
        
        sum_synapse_1 = np.dot(hidden_layer, self.weights_1) + self.biases_2
        output_layer = self._act_func(sum_synapse_1)
        
        # Return everything
        return X, hidden_layer, output_layer
    

    def _back_propagation(self, output_layer):
        error_output_layer = self.y_np_train - output_layer
        average_error = np.mean(abs(error_output_layer))
        
        derivative_output = self._act_func_deriv(output_layer)
        delta_output = error_output_layer * derivative_output

        return delta_output, average_error
    
    
    def _back_propagation_hidden(self, delta_output, hidden_layer):
    
        # Transpose the numpy array
        weights_1T = self.weights_1.T
        
        delta_output_weight = delta_output.dot(weights_1T)
        delta_hidden_layer = delta_output_weight * self._act_func_deriv(hidden_layer)
        
        hidden_layerT = hidden_layer.T
        input_x_delta1 = hidden_layerT.dot(delta_output)

        return delta_hidden_layer, input_x_delta1
    

    def _weight_bias_update(self, input_layer, input_x_delta1, delta_hidden_layer, delta_output):
        #global weights_0, weights_1, biases_1, biases_2, learning_rate

        m = input_layer.shape[0]                                                          # Batch size to normalise the gradient
        
        self.weights_1 = self.weights_1 + (input_x_delta1 / m * self.learning_rate)       # Divide by m to compute the average gradient
        
        input_layerT = input_layer.T
        input_x_delta0 = input_layerT.dot(delta_hidden_layer)
        self.weights_0 = self.weights_0 + (input_x_delta0 / m * self.learning_rate)                                                            

        self.biases_1 = self.biases_1 + np.sum(delta_hidden_layer, axis=0, keepdims=True) / m * self.learning_rate
        self.biases_2 = self.biases_2 + np.sum(delta_output, axis=0, keepdims=True) / m * self.learning_rate 


    def calculate_accuracy(self, X, y):
        _, _, output = self._feed_forward(X)
        predictions = (output > 0.5).astype(int)
        return np.mean(predictions == y) * 100

    # ***** Public Functions 

    def train_network_old (self, epochs, **kwargs):
        
        # Set up parameters
        self.learning_rate = kwargs.get('learning_rate', self.learning_rate)
        self.error = []

        # Loop through the training epochs
        for epoch in range(epochs):
            input_layer, hidden_layer, output_layer = self._feed_forward(self.X_np_train)       # Calling feedForward method

            delta_output, average_error = self._back_propagation(output_layer)                                             # Calling backPropagation method

            if epoch % 100 == 0:
                self.error.append(average_error)
                if self.debug:
                    print_output('Epoch', str(epoch + 1) + " ", newline=False)
                    print_output('Error', str(average_error))
                

            delta_hidden_layer, input_x_delta1 = self._back_propagation_hidden(delta_output, hidden_layer)                # Calling backPropagationHidden method

            self._weight_bias_update(input_layer, input_x_delta1, delta_hidden_layer, delta_output)                       # Calling weightBiasUpdate method

    def train_network(self, epochs, **kwargs):

        # Set up parameters
        self.learning_rate = kwargs.get('learning_rate', self.learning_rate)
        self.train_error_history = []
        self.valid_error_history = [] # To track validation

        for epoch in range(epochs):

            # Forward
            input_layer, hidden_layer, output_layer = self._feed_forward(self.X_np_train)
            
            # Backprop (Output)
            delta_output, train_error = self._back_propagation(output_layer)
            
            # Backprop (Hidden)
            delta_hidden_layer, input_x_delta1 = self._back_propagation_hidden(delta_output, hidden_layer)
            
            # Update Weights
            self._weight_bias_update(input_layer, input_x_delta1, delta_hidden_layer, delta_output)


            if epoch % 100 == 0:
                # We use the SAME feed_forward method but with TEST data
                _, _, test_output = self._feed_forward(self.X_np_test)
                
                # Calculate validation error (Mean Absolute Error)
                valid_error = np.mean(np.abs(self.y_np_test - test_output))
                
                self.train_error_history.append(train_error)
                self.valid_error_history.append(valid_error)

                if self.debug:
                    print(f"Epoch {epoch}: Train Error: {train_error:.4f} | Valid Error: {valid_error:.4f}")
                    

    def evaluate(self, X_data=None, y_data=None):
            
        # Default to the test set if no specific data is provided
        if X_data is None:
            X_data = self.X_np_test
            y_data = self.y_np_test
                
        # 1. Run the forward pass (Prediction)
        _, _, output_layer = self._feed_forward(X_data)
           
        # 2. Convert probabilities to binary 0 or 1
        # If the network says 0.8, we count that as "Yes" (1)
        predictions = (output_layer > 0.5).astype(int)
            
        # 3. Compare with actual y values
        accuracy = np.mean(predictions == y_data) * 100
            
        return accuracy

    def plot_error(self):
        plt.figure(figsize=(10, 6))
        plt.plot(self.train_error_history, label='Training Error')
        plt.plot(self.valid_error_history, label='Validation Error', linestyle='--')
        plt.xlabel('Epochs (x100)')
        plt.ylabel('Mean Absolute Error')
        plt.title('Training vs Validation Error')
        plt.legend()
        plt.show()


    def get_confusion_matrix(y_true, y_pred_probs):
        # Convert probabilities to 0 or 1
        y_pred = (y_pred_probs > 0.5).astype(int)
        
        # Ensure they are flat arrays for easy comparison
        y_true = y_true.flatten()
        y_pred = y_pred.flatten()
        
        tp = np.sum((y_true == 1) & (y_pred == 1))
        tn = np.sum((y_true == 0) & (y_pred == 0))
        fp = np.sum((y_true == 0) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        
        return np.array([[tn, fp], [fn, tp]])
    
    def plot_confusion_matrix(matrix):
        fig, ax = plt.subplots(figsize=(6, 5))
        im = ax.imshow(matrix, cmap='Blues')

        # Add colorbar
        plt.colorbar(im)

        # Set labels
        classes = ['No (0)', 'Yes (1)']
        ax.set_xticks(np.arange(len(classes)))
        ax.set_yticks(np.arange(len(classes)))
        ax.set_xticklabels(classes)
        ax.set_yticklabels(classes)

        # Rotate the tick labels and set their alignment.
        plt.setp(ax.get_xticklabels(), rotation=0, ha="center")

        # Loop over data dimensions and create text annotations.
        for i in range(2):
            for j in range(2):
                text = ax.text(j, i, matrix[i, j],
                            ha="center", va="center", color="black", fontsize=14)

        ax.set_title("Confusion Matrix Heatmap")
        ax.set_xlabel('Predicted Label')
        ax.set_ylabel('True Label')
        plt.tight_layout()
        plt.show()

    # Text Representation
    def print_confusion_matrix(matrix):
        print("\n--- Confusion Matrix (Text Table) ---")
        print(f"{'':>15} | {'Predicted No':>12} | {'Predicted Yes':>12} |")
        print("-" * 48)
        print(f"{'Actual No':>15} | {matrix[0][0]:>12} | {matrix[0][1]:>12} |")
        print(f"{'Actual Yes':>15} | {matrix[1][0]:>12} | {matrix[1][1]:>12} |")
        print("-" * 48)

    # Export the model weights and biases to a JSON file
    def save_weights(self, filename):
        weights_data = {
            'weights_0': self.weights_0.tolist(),
            'weights_1': self.weights_1.tolist(),
            'biases_1': self.biases_1.tolist(),
            'biases_2': self.biases_2.tolist()
        }
        with open(filename, 'w') as f:
            json.dump(weights_data, f)

    # Load the model weights and biases from a JSON file
    def load_weights(self, filename):
        with open(filename, 'r') as f:
            weights_data = json.load(f)
            self.weights_0 = np.array(weights_data['weights_0'])
            self.weights_1 = np.array(weights_data['weights_1'])
            self.biases_1 = np.array(weights_data['biases_1'])
            self.biases_2 = np.array(weights_data['biases_2'])
        

This is a binary classification problem, which implies 1 output neuron

In [ ]:
mlp = MLP_From_Scratch(df_X_train, df_y_train, df_X_test, df_y_test, debug=True)
mlp.train_network(epochs=2000)
mlp.plot_error()


In [ ]:
mlp.calculate_accuracy(mlp.X_np_test, mlp.y_np_test)

In [ ]:
final_test_score = mlp.evaluate()
print(f"Final Model Accuracy on Unseen Data: {final_test_score:.2f}%")

In [ ]:
sys.exit('Stopping notebook run here.')   

Notes for report:

Why use a Validation set instead of just the Test set?

If you use your Test set to decide when to stop training or what learning rate to use, you are technically "leaking" information. You might end up with a model that performs great on that specific Test set but fails in the real world. By using a Validation set, you keep the Test set completely "unseen" until the very final moment of your project.


Three Crucial "Scratch" Tips for your Assessment:

1. The "Sign" of your Update: In your _weight_bias_update, you are using:
self.weights_1 = self.weights_1 + (input_x_delta1 ...)
Normally, Gradient Descent subtracts the gradient. However, because you calculated error_output_layer as Target - Output (which is the negative gradient), using + is mathematically correct here. Just keep an eye on that!

2. Accuracy vs. Error:
Error (MAE) is great for the math, but your assessment likely wants to see Accuracy. You can add a quick method to calculate how many predictions were correct:

To know how "wrong" the network is, we use Binary Cross-Entropy Loss (also called Log Loss). This is much more effective for classification than Mean Squared Error.
Loss=−m1​∑[y⋅log(y^​)+(1−y)⋅log(1−y^​)]

In [ ]:
def compute_loss(y_true, y_pred):
    m = y_true.shape[0]
    # Adding a tiny epsilon (1e-15) prevents log(0) which would result in NaN
    loss = -1/m * np.sum(y_true * np.log(y_pred + 1e-15) + (1 - y_true) * np.log(1 - y_pred + 1e-15))
    return loss

In [ ]:
def back_propagation(X, y, Z1, A1, Z2, A2, W1, W2):
    m = X.shape[0]
    
    # 1. Output Layer Gradients
    dZ2 = A2 - y  # Error signal at the end
    dW2 = (1/m) * np.dot(A1.T, dZ2)
    db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)
    
    # 2. Hidden Layer Gradients
    # We pass the error back through W2 and then through the Sigmoid derivative
    dZ1 = np.dot(dZ2, W2.T) * act_func_deriv(A1)
    dW1 = (1/m) * np.dot(X.T, dZ1)
    db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)
    
    return dW1, db1, dW2, db2

In [ ]:
def train_network(X, y, epochs=1000, learning_rate=0.1):
    # 1. Initialize Weights with the Seed (Reproduction)
    np.random.seed(42)
    input_size = X.shape[1]
    hidden_size = 16
    output_size = 1
    
    W1 = np.random.randn(input_size, hidden_size) * 0.01
    b1 = np.zeros((1, hidden_size))
    W2 = np.random.randn(hidden_size, output_size) * 0.01
    b2 = np.zeros((1, output_size))
    
    loss_history = []

    for i in range(epochs):
        # 2. Forward Propagation
        Z1, A1, Z2, A2 = forward_propagation(X, W1, b1, W2, b2)
        
        # 3. Compute and store Loss
        current_loss = compute_loss(y, A2)
        loss_history.append(current_loss)
        
        # 4. Backpropagation (Calculate Gradients)
        dW1, db1, dW2, db2 = back_propagation(X, y, Z1, A1, Z2, A2, W1, W2)
        
        # 5. Gradient Descent (Update Weights)
        # New Weight = Old Weight - (Learning Rate * Gradient)
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
        
        # 6. Print progress every 100 epochs
        if i % 100 == 0:
            print(f"Epoch {i}: Loss {current_loss:.4f}")
            
    return W1, b1, W2, b2, loss_history

In [ ]:
# Train the MLP
W1_final, b1_final, W2_final, b2_final, history = train_network(X_np_train, y_np_train, epochs=2000, learning_rate=0.5)

# Final Accuracy Check
_, _, _, final_preds = forward_propagation(X_np_train, W1_final, b1_final, W2_final, b2_final)
final_accuracy = np.mean((final_preds > 0.5) == y_np_train) * 100
print(f"\nFinal Accuracy: {final_accuracy:.2f}%")

In [ ]:
# Use the final weights to predict on the TEST set
_, _, _, test_probs = forward_propagation(X_np_test, W1_final, b1_final, W2_final, b2_final)

# Convert probabilities to 0 or 1
test_preds = (test_probs > 0.5).astype(int)

# Calculate Accuracy
test_accuracy = np.mean(test_preds == y_np_test) * 100
print(f"Test Accuracy: {test_accuracy:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history)
plt.title('Model Loss Over Time')
plt.xlabel('Epochs')
plt.ylabel('Binary Cross-Entropy Loss')
plt.show()

---